# Which Orders Come Back?

Every return costs the store twice: the sale is gone and the shipping, both
legs of it, was spent for nothing. Right now a return is only visible after it
happens. This notebook asks whether it is visible earlier: given what is known
about an order at the point it would be worth a second look, can a model flag
the ones likely to come back, so a review queue can be built around risk
instead of running blind?

### What this notebook concludes, up front

This one ends in a negative result, and the negative result is the
deliverable.

Returns cannot be predicted usefully from this data. The best model reaches
**0.132** average precision against a **0.047** base rate, about three times
better than chance at ranking and nowhere near enough to act on per order. A
plain logistic regression matches a tuned gradient-boosted tree, which is
itself a signal: when extra model capacity buys nothing, there is usually
nothing left to find.

The more useful finding is why. Three of the seven markets, covering 4,893
orders over four full years, record **zero** returns. Not a low rate, zero.
The four markets that do record returns cluster tightly between 5.4% and 6.2%,
exactly what one process running everywhere would look like. The probability
of EMEA's 2,462 orders producing no returns at the store-wide rate is about 10
to the power of -52. This is a gap in how returns were collected, not a fact
about customers, and it is the single strongest feature the model has.

That matters well beyond this notebook. Any dashboard or report that breaks
returns down by geography is wrong right now, and would show Africa and EMEA
as flawless operations. Fixing the extract is worth more than any model built
on top of it.

Section 5 lists what would actually be needed to answer the question:
per-product and per-customer return history, and reason codes. None of it is
in this warehouse.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = next(p for p in Path.cwd().parents if (p / "utils").is_dir())
sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import shap
from IPython.display import Markdown, display
from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, f1_score, precision_score, recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight

from utils import custom_plots as cp
from utils import custom_stats as cs
from utils.db_utils import run_query

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 50)

D:\miniconda3\envs\analyst_313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Pulling the order-grain data

A return is recorded once per order in the source data, not once per line: an
order either came back or it didn't. The original notebook trained on
`Fact_Sales` (one row per order line), which repeats a single order's label
across every line it has, inflates the sample from 25k to 50k rows, and can
put the same order on both sides of a train/test split. Everything here comes
from `olap.fact_order` instead: one row per order, 25,033 rows.

Product category isn't part of `fact_order`, because an order can span several
products, so it has no `product_key` of its own. The subqueries below roll
`fact_sales` up to the order to recover it: which category the order's money
mostly went to, and how many distinct categories it touched.

In [3]:
df = run_query("""
    WITH cat_sales AS (
        SELECT fs.order_id, p.category, SUM(fs.sales) AS cat_sales
        FROM olap.fact_sales fs
        JOIN olap.dim_product p ON p.product_key = fs.product_key
        GROUP BY fs.order_id, p.category
    ),
    order_category AS (
        SELECT DISTINCT ON (order_id) order_id, category AS primary_category
        FROM cat_sales
        ORDER BY order_id, cat_sales DESC
    ),
    order_cat_count AS (
        SELECT fs.order_id, COUNT(DISTINCT p.category) AS category_count
        FROM olap.fact_sales fs
        JOIN olap.dim_product p ON p.product_key = fs.product_key
        GROUP BY fs.order_id
    )
    SELECT
        f.order_id,
        f.is_returned,
        f.line_count,
        f.product_count,
        f.quantity,
        f.sales,
        f.discount_rate,
        f.profit_margin,
        f.shipping_cost_pct,
        f.ship_lag_days,
        sm.ship_mode,
        pr.priority,
        cu.segment,
        g.market,
        g.region,
        od.day_name AS order_dow,
        od.month    AS order_month,
        oc.category_count,
        ocat.primary_category
    FROM olap.fact_order f
    JOIN olap.dim_ship_mode      sm   ON sm.ship_mode_key = f.ship_mode_key
    JOIN olap.dim_order_priority pr   ON pr.priority_key  = f.priority_key
    JOIN olap.dim_customer       cu   ON cu.customer_key  = f.customer_key
    JOIN olap.dim_geography      g    ON g.geo_key        = f.geo_key
    JOIN olap.dim_order_date     od   ON od.date_key      = f.order_date_key
    JOIN order_cat_count  oc   ON oc.order_id   = f.order_id
    JOIN order_category   ocat ON ocat.order_id = f.order_id
    ORDER BY f.order_id
""")
df["order_month"] = df["order_month"].astype(int)
df.shape

(25033, 19)

In [4]:
assert df.isna().sum().sum() == 0, "unexpected nulls in the pulled frame"
assert df["order_id"].is_unique, "expected one row per order"
df.head()

,order_id,is_returned,line_count,product_count,quantity,sales,discount_rate,profit_margin,shipping_cost_pct,ship_lag_days,ship_mode,priority,segment,market,region,order_dow,order_month,category_count,primary_category
0,AE-2011-9160,False,2,2,8,161.082,0.7,-1.527657,0.059349,4,Standard Class,Medium,Consumer,EMEA,EMEA,Monday,10,2,Office Supplies
1,AE-2013-1130,False,2,2,7,228.996,0.7,-1.034795,0.262799,0,Same Day,High,Consumer,EMEA,EMEA,Monday,10,2,Furniture
2,AE-2013-1530,False,2,2,3,23.634,0.7,-1.611069,0.133706,3,Second Class,High,Corporate,EMEA,EMEA,Tuesday,12,1,Office Supplies
3,AE-2014-2840,False,1,1,1,42.480,0.7,-1.766949,0.189266,3,First Class,Critical,Consumer,EMEA,EMEA,Wednesday,11,1,Office Supplies
4,AE-2014-3830,False,6,6,16,281.502,0.7,-1.524352,0.068845,6,Standard Class,Medium,Consumer,EMEA,EMEA,Saturday,12,2,Office Supplies


## 2. How rare is a return

This number decides every metric choice that follows, so it comes before
anything else.

In [5]:
n_returned = int(df["is_returned"].sum())
n_total = len(df)
cs.proportion_ci(n_returned, n_total, labels=["returned"])

,label,successes,n,proportion,ci_low,ci_high,width,method
0,returned,1172,25033,0.046818,0.04427,0.049505,0.005235,wilson


4.68% of orders come back. That is the base rate a model has to beat, and it
rules accuracy out on its own: a rule that never flags a return is right 95.3%
of the time while catching none of them. The metrics used from here on are
precision, recall and average precision, the area under the precision-recall
curve, because those are the ones that still say something when the positive
class is this thin.

In [6]:
# strategy="most_frequent" ignores X entirely; the column passed is a
# placeholder, not a feature the model is allowed to use.
dummy_preview = DummyClassifier(strategy="most_frequent")
dummy_preview.fit(df[["quantity"]], df["is_returned"])
preview_pred = dummy_preview.predict(df[["quantity"]])

print(f"accuracy predicting 'never returned':  {(preview_pred == df['is_returned']).mean():.1%}")
print(f"recall on actual returns:              {(preview_pred[df['is_returned']] == True).mean():.1%}")

accuracy predicting 'never returned':  95.3%
recall on actual returns:              0.0%


95.3% accuracy from a rule that catches zero returns is exactly the trap. A
model report that leads with accuracy on this problem is not saying anything.

## 3. Which fields actually move with a return

A quick association scan before building anything: which categorical fields
carry a real relationship to `is_returned`, and how confident can that be
given how lopsided the target is.

In [7]:
assoc = pd.concat([
    cs.association_test(df, "market", "is_returned"),
    cs.association_test(df, "region", "is_returned"),
    cs.association_test(df, "ship_mode", "is_returned"),
    cs.association_test(df, "priority", "is_returned"),
    cs.association_test(df, "segment", "is_returned"),
    cs.association_test(df, "primary_category", "is_returned"),
], ignore_index=True)
assoc[["variables", "n", "statistic", "p_value", "effect_size", "magnitude"]]

,variables,n,statistic,p_value,effect_size,magnitude
0,market × is_returned,25033,302.035177,2.996892e-62,0.108749,small
1,region × is_returned,25033,824.336991,9.976474e-169,0.180144,small
2,ship_mode × is_returned,25033,4.049007,2.562207e-01,0.006473,negligible
3,priority × is_returned,25033,1.445103,6.949973e-01,0.000000,negligible
4,segment × is_returned,25033,2.799699,2.466340e-01,0.005652,negligible
5,primary_category × is_returned,25033,23.514103,7.833888e-06,0.029317,negligible


Market and region separate from the rest by a wide margin; everything else
sits in negligible territory even with 25,033 rows behind each test. That is
worth looking at before it goes anywhere near a model.

## 4. A wrinkle in the geography

Market's effect size is large enough to check by hand rather than take on
faith.

In [8]:
market_rates = df.groupby("market")["is_returned"].agg(["sum", "count"])
market_rates.columns = ["n_returned", "n_orders"]
market_rates["pct_returned"] = (market_rates["n_returned"] / market_rates["n_orders"] * 100).round(2)
market_rates.sort_values("pct_returned", ascending=False)

,n_returned,n_orders,pct_returned
market,,,
EU,284,4593,6.18
US,297,4990,5.95
LATAM,295,5120,5.76
APAC,296,5437,5.44
Africa,0,2230,0.00
Canada,0,201,0.00
EMEA,0,2462,0.00


In [9]:
df["is_returned_num"] = df["is_returned"].astype(int)
cp.grouped_bar_plot(
    df, group_col="market", value_col="is_returned_num", agg="mean",
    ci_method="bootstrap", n_boot=2000,
    title="Share of Orders Returned, by Market",
)

Four years of orders, and Africa, EMEA and Canada, 4,893 orders and close to a
fifth of the book, show not one recorded return between them, next to 5.4-6.2%
in the other four markets. A genuine zero would need a written no-returns
policy that held perfectly for thousands of orders across three unrelated
markets over four years; nothing in the warehouse documents that, and it reads
far more like a gap in how the returns feed covers geography than a real
behavioural difference. There's no table here that settles which it is; that
question belongs to whoever owns the source extract.

Two things follow. First, `market` and `region` collide on their labels:
"Central", "North" and "South" each appear inside more than one market, at
different rates (EU Central runs 6.3% returned, LATAM Central runs 3.1%), so
neither column can be used alone without folding unrelated places together.
The cell below builds a combined field that keeps them distinct. Second, and
more important: because this pattern is more likely a data artefact than a
business fact, whatever the model learns from geography needs to be checked
against the rest of the book before it's trusted, and that check is Section
11.

In [10]:
df["market_region"] = df["market"] + " - " + df["region"]
region_counts = df["market_region"].value_counts()
print(f"{region_counts.size} market_region combinations, "
      f"smallest holds {region_counts.min()} orders")

18 market_region combinations, smallest holds 201 orders


## 5. What's off-limits

A return is confirmed after delivery, so the question is what's genuinely
known about an order by the time it would be worth flagging, not what's known
when it's placed. That's a looser bar than usual, and it's why `ship_lag_days`
stays in here: on the ship-mode model elsewhere in this project it was dropped
for being a consequence of the thing being predicted, but a return doesn't
cause a slow shipment; if anything the arrow points the other way, a slow or
troubled delivery is a plausible *cause* of a return. Dropped instead:

- **`order_count`, `active_span_days`, `first_order_date`,
  `last_order_date`** (customer dimension), lifetime aggregates over a
  customer's entire history in the warehouse, 2011 through 2014. For an
  order placed in 2011, `order_count` already counts orders that customer
  placed in 2013. That's the future leaking into a feature for a
  prediction made in the past, so none of the four were even pulled into
  the query above.
- **`gross_sales`, `discount_amount`, `cost`**, fixed arithmetic on
  columns already kept (`sales`, `discount_rate`, `profit_margin`).
  Keeping both sides of an identity adds correlated noise, not
  information, so these weren't pulled either.
- **`ship_date_key` / `dim_ship_date`**, the calendar detail of the ship
  date itself (day of week, month) adds nothing past what `ship_lag_days`
  already summarises as a single number, and pulling the same event twice
  through two paths just duplicates it.

One more question is worth asking directly rather than assuming an answer:
does `profit`, `profit_margin` or `discount_rate` change once an order is
returned? They don't. The warehouse's returns source is a plain list of order
IDs, a flag joined onto the order rather than a rewrite of its financials, so
these three describe the sale as it was transacted, before any return outcome
existed. They're money the store already recorded, not money it recorded
knowing the order would come back.

### Final feature set

In [11]:
categorical_features = [
    "ship_mode", "priority", "segment", "market_region",
    "order_dow", "order_month", "primary_category",
]
numeric_features = [
    "line_count", "product_count", "quantity", "sales", "discount_rate",
    "profit_margin", "shipping_cost_pct", "ship_lag_days", "category_count",
]

pd.DataFrame({
    "feature": categorical_features + numeric_features,
    "kind": (["categorical"] * len(categorical_features)
             + ["numeric"] * len(numeric_features)),
})

,feature,kind
0,ship_mode,categorical
1,priority,categorical
2,segment,categorical
3,market_region,categorical
4,order_dow,categorical
5,order_month,categorical
6,primary_category,categorical
7,line_count,numeric
8,product_count,numeric
9,quantity,numeric


## 6. Splitting the data

Nothing about this question changes with the calendar. An order from 2011
isn't more or less predictable than one from 2014, so there's no forecasting
case for holding out the newest orders. What matters instead is that the 4.68%
who return still land in roughly that share in every split, which a stratified
split guarantees.

Three-way split: train to fit, validation to decide when the boosted tree
stops adding rounds, test touched exactly once for the numbers reported below.

In [12]:
feature_cols = categorical_features + numeric_features
X = df[feature_cols].copy()
y = df["is_returned"].astype(int)

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.25, stratify=y_train_full, random_state=42
)
pd.Series({"train": len(X_train), "validation": len(X_val), "test": len(X_test)})

train         15019
validation     5007
test           5007
dtype: int64

## 7. Baselines

Two trivial models before anything with a learning curve, scored on the same
held-out test set every later model will use.

In [13]:
def score_row(name: str, y_pred: np.ndarray, y_score: np.ndarray) -> dict:
    """Precision, recall, F1 and average precision for one set of predictions."""
    return {
        "model": name,
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "average_precision": average_precision_score(y_test, y_score),
    }

In [14]:
dummy = DummyClassifier(strategy="most_frequent", random_state=42)
dummy.fit(X_train, y_train)
dummy_pred = dummy.predict(X_test)
dummy_score = dummy.predict_proba(X_test)[:, 1]

In [15]:
preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ("num", StandardScaler(), numeric_features),
])
logreg_plain = Pipeline([
    ("prep", preprocessor),
    ("clf", LogisticRegression(max_iter=1000, random_state=42)),
])
logreg_plain.fit(X_train, y_train)
logreg_plain_pred = logreg_plain.predict(X_test)
logreg_plain_score = logreg_plain.predict_proba(X_test)[:, 1]

baseline_scores = pd.DataFrame([
    score_row("dummy (most frequent)", dummy_pred, dummy_score),
    score_row("logistic regression (unweighted)", logreg_plain_pred, logreg_plain_score),
]).set_index("model")
baseline_scores.round(4)

,precision,recall,f1,average_precision
model,,,,
dummy (most frequent),0.0,0.0,0.0,0.0467
logistic regression (unweighted),0.0,0.0,0.0,0.1379


The dummy model can't earn a recall above zero, because it never predicts a
return. The unweighted logistic regression predicts a handful, but with no
correction for class size the fit spends nearly all its effort on the majority
class and recall stays low. Average precision is the number to watch across
every row in this table; it doesn't depend on picking a threshold the way
precision and recall do.

## 8. Class weights, not synthetic oversampling

The original notebook ran `SMOTE(k_neighbors=5)` on the training fold and
trained every model on the resampled data. Two problems with that. First, it
manufactures order records that never existed. At a 4.68% positive rate SMOTE
mostly interpolates between positives that are already each other's nearest
neighbours, which invents very little genuinely new information. Second, and
worse, the resampling happened before the GridSearchCV split the data into
folds, so synthetic points built from a validation fold's own neighbours could
leak into that same fold's evaluation.

Reweighting the real rows sidesteps both: it's one parameter, invents nothing,
and there's no ordering to get wrong because there's no separate resampling
step to place badly.

In [16]:
class_weights = dict(zip(
    [0, 1], compute_class_weight("balanced", classes=np.array([0, 1]), y=y_train)
))
pd.Series(class_weights, name="weight")

0     0.524553
1    10.682077
Name: weight, dtype: float64

In [17]:
logreg_weighted = Pipeline([
    ("prep", preprocessor),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
])
logreg_weighted.fit(X_train, y_train)
logreg_weighted_pred = logreg_weighted.predict(X_test)
logreg_weighted_score = logreg_weighted.predict_proba(X_test)[:, 1]

weight_effect = pd.DataFrame([
    score_row("logistic regression (unweighted)", logreg_plain_pred, logreg_plain_score),
    score_row("logistic regression (weighted)", logreg_weighted_pred, logreg_weighted_score),
]).set_index("model")
weight_effect.round(4)

,precision,recall,f1,average_precision
model,,,,
logistic regression (unweighted),0.0000,0.0000,0.000,0.1379
logistic regression (weighted),0.0861,0.6453,0.152,0.1391


Weighting trades precision for recall, visibly: the model stops treating the
4.68% minority as noise to average away, at the cost of more false alarms.
Average precision, which doesn't move with the threshold, is the fairer read
on whether the trade helped.

## 9. LightGBM

LightGBM takes categorical columns natively (cast to pandas `category` dtype,
no one-hot encoding needed), which is also why there's no `StandardScaler`
here: a boosted tree splits on thresholds, not distances, so scaling the
numeric columns would change nothing about what it can learn. That's the bug
named in Section 8's original notebook comparison solved differently. The
original *did* scale for its tree model, by running the same log-and-MinMax
transformer through every feature regardless of model type, and that
transformer's own bug (its `MinMaxScaler` was refit inside `.transform()`, so
test rows were scaled against their own range instead of the training range)
never had a chance to matter for a model that didn't need scaling in the first
place. Neither problem exists here: nothing here is scaled, and there is no
second fit hiding inside a transform.

In [18]:
X_train_c = X_train.copy()
X_val_c = X_val.copy()
X_test_c = X_test.copy()
for c in categorical_features:
    X_train_c[c] = X_train_c[c].astype("category")
    X_val_c[c] = pd.Categorical(X_val_c[c], categories=X_train_c[c].cat.categories)
    X_test_c[c] = pd.Categorical(X_test_c[c], categories=X_train_c[c].cat.categories)

# Library defaults, and early stopping left to watch whatever LightGBM watches
# by default. Kept in the notebook because what it does next is instructive.
lgbm_naive = LGBMClassifier(
    n_estimators=2000, learning_rate=0.05, num_leaves=31,
    class_weight="balanced", random_state=42, verbosity=-1,
)
lgbm_naive.fit(
    X_train_c, y_train, eval_set=[(X_val_c, y_val)],
    callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)],
)
print(f"stopped at round {lgbm_naive.best_iteration_} of 2000")

stopped at round 896 of 2000


In [19]:
def ap_by_split(model) -> pd.DataFrame:
    # Average precision on all three splits. Memorisation is invisible on the
    # test score alone; it shows up as the distance from the training score.
    return pd.DataFrame([
        {"split": "train", "average_precision": average_precision_score(
            y_train, model.predict_proba(X_train_c)[:, 1])},
        {"split": "validation", "average_precision": average_precision_score(
            y_val, model.predict_proba(X_val_c)[:, 1])},
        {"split": "test", "average_precision": average_precision_score(
            y_test, model.predict_proba(X_test_c)[:, 1])},
    ]).set_index("split")

ap_by_split(lgbm_naive).round(4)

,average_precision
split,
train,1.0000
validation,0.0981
test,0.0986


A training average precision of 1.0000 against 0.098 on validation is not a
model, it is a lookup table. Two things caused it, and both are worth naming.

Early stopping was watching the wrong quantity. `LGBMClassifier` scores its
`eval_set` with binary logloss unless told otherwise, and logloss keeps
improving on the majority class long after the ranking of the rare class has
stopped improving, so the run went 896 rounds instead of stopping early.
Naming `average_precision` in `fit` puts the stopping rule on the metric the
problem is actually judged by.

The capacity was also far too high for the evidence available. 31 leaves per
tree over 896 rounds, fitted to 703 positive orders, gives the model more than
enough room to isolate individual training rows. The refit below reduces it to
7 leaves, requires 60 orders behind every leaf, and subsamples both rows and
columns on each round.

In [20]:
lgbm = LGBMClassifier(
    n_estimators=3000, learning_rate=0.03, num_leaves=7, min_child_samples=60,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.7, reg_lambda=5.0,
    class_weight="balanced", random_state=42, verbosity=-1,
)
lgbm.fit(
    X_train_c, y_train, eval_set=[(X_val_c, y_val)],
    eval_metric="average_precision",
    callbacks=[lgb.early_stopping(150, verbose=False), lgb.log_evaluation(0)],
)
print(f"stopped at round {lgbm.best_iteration_} of 3000")

lgbm_split_scores = ap_by_split(lgbm)
lgbm_split_scores.round(4)

stopped at round 98 of 3000


,average_precision
split,
train,0.1644
validation,0.1607
test,0.1324


In [21]:
lgbm_pred = lgbm.predict(X_test_c)
lgbm_score = lgbm.predict_proba(X_test_c)[:, 1]

model_scores = pd.concat([
    baseline_scores,
    weight_effect.loc[["logistic regression (weighted)"]],
    pd.DataFrame([score_row("lightgbm", lgbm_pred, lgbm_score)]).set_index("model"),
])
model_scores.round(4)

,precision,recall,f1,average_precision
model,,,,
dummy (most frequent),0.0000,0.0000,0.0000,0.0467
logistic regression (unweighted),0.0000,0.0000,0.0000,0.1379
logistic regression (weighted),0.0861,0.6453,0.1520,0.1391
lightgbm,0.0865,0.7051,0.1541,0.1324


Read the average-precision column and set the other three aside for a moment.
The unweighted logistic model scores 0.138 there while posting zeros for
precision, recall and F1. It ranks orders perfectly usefully; it simply never
pushes a single one past 0.5, so any metric built on `predict()` records it as
having done nothing at all. That is the imbalance problem in one row, and it
is why every decision from here on is made on the ranking plus a chosen
cut-off rather than on the library default.

Against a base rate of 0.047, every model that uses the features lands near
0.13 to 0.14, roughly three times better than chance at ordering the book by
risk. Class weighting transforms what `predict()` returns, moving recall from
0.00 to 0.65, while barely touching average precision: 0.138 to 0.139.
Weighting moves the threshold, not the ranking.

The booster does not win. The next cell checks whether the gap between it and
the linear model is real or noise.

In [22]:
# Is the booster actually better than the linear model, or is the gap noise?
# Resample the test set and re-score both on each draw.
rng = np.random.default_rng(0)
y_arr = y_test.to_numpy()
logreg_score = logreg_weighted.predict_proba(X_test)[:, 1]

ap_gaps = []
for _ in range(2000):
    draw = rng.integers(0, len(y_arr), len(y_arr))
    if y_arr[draw].sum() == 0:
        continue
    ap_gaps.append(average_precision_score(y_arr[draw], lgbm_score[draw])
                   - average_precision_score(y_arr[draw], logreg_score[draw]))

gap_lo, gap_hi = np.percentile(ap_gaps, [2.5, 97.5])
print(f"average precision, lightgbm minus logistic: {np.mean(ap_gaps):+.4f}")
print(f"95% bootstrap interval: {gap_lo:+.4f} to {gap_hi:+.4f}")

average precision, lightgbm minus logistic: -0.0069
95% bootstrap interval: -0.0281 to +0.0111


The interval straddles zero, so the two models are indistinguishable on this
test set. A linear model over one-hot features matches a gradient-boosted tree
with native categorical splits and 98 rounds of fitting.

That is worth pausing on rather than moving past. When additional capacity
buys nothing, the usual reason is that there is little non-linear structure
left to find, and every later section of this notebook points the same way.
The booster carries on below only because it was selected on the validation
split; the linear model would serve just as well.

## 10. Evaluating the ranking

In [23]:
cp.classification_curve_plot(
    y_test, lgbm_score, kind="pr",
    title="Precision-Recall — LightGBM, Test Set",
)

The curve starts high and falls away fast. Among the handful of orders the
model is most confident about, precision runs above 20%; by the time recall
reaches half the returns it has decayed to about 10%. The dashed reference is
the 4.7% base rate, so the ranking beats chance across the whole range.

Average precision of 0.132 against that 0.047 base rate is the honest summary,
and it says two things at once. There is real signal here, about threefold at
the left end of the curve. It is also a weak model: four out of five of the
orders it is most sure about still do not come back.

In [24]:
cp.classification_curve_plot(
    y_test, lgbm_score, kind="roc",
    title="ROC — LightGBM, Test Set",
)

ROC holds the diagonal as its reference regardless of the class split, so it
flatters a rare-positive problem like this one, since a model can look strong
on ROC-AUC mostly by being better than chance on the easy 95.3% majority. The
PR curve's reference is the base rate itself, which is why it's the fairer
read when returns are this scarce.

In [25]:
cp.calibration_plot(
    y_test, lgbm_score,
    title="Calibration — LightGBM Predicted Return Probability",
)

These scores are not probabilities, and the plot says so plainly. Predictions
average 0.40 while returns actually happen 4.7% of the time, so every point
sits far below the diagonal. This is `class_weight="balanced"` doing exactly
what it was asked to: re-weighting the classes drags the decision boundary
toward the rare class and inflates the scores by roughly the weight ratio.

That does not invalidate the model, but it constrains how the output may be
used. "Flag every order above 40% return risk" is not a sentence this model
can support, because 40% here does not mean 40%. "Flag the riskiest 5% of
orders" is fine, because the ordering survives the distortion even though the
scale does not. Every operating point below is therefore defined as a share of
orders, never as a probability threshold.

## 11. Is the market gap doing all the work?

Section 4 flagged something the model can't tell apart from real behaviour:
three markets with a suspicious, total absence of recorded returns. If the
model's score is mostly "which market is this," it will look strong overall
while adding nothing where a return could actually happen. The test below
restricts to the four markets that show any return history at all (EU, US,
LATAM, APAC) and re-scores there.

In [26]:
zero_return_markets = ["Africa", "EMEA", "Canada"]
test_market = df.loc[X_test.index, "market"]
active_mask = ~test_market.isin(zero_return_markets)

ap_full = average_precision_score(y_test, lgbm_score)
ap_active = average_precision_score(y_test[active_mask], lgbm_score[active_mask])
base_rate_active = y_test[active_mask].mean()

pd.DataFrame({
    "subset": ["full test set", "active-return markets only"],
    "n": [len(y_test), int(active_mask.sum())],
    "base_rate": [y_test.mean(), base_rate_active],
    "average_precision": [ap_full, ap_active],
}).round(4)

,subset,n,base_rate,average_precision
0,full test set,5007,0.0467,0.1324
1,active-return markets only,4000,0.0585,0.1325


The score still works where returns are possible, but the advantage shrinks
once the easy part of the job is taken away. Restricted to EU, US, LATAM and
APAC, average precision is essentially unchanged at 0.132 while the base rate
rises from 4.7% to 5.9%. Lift over chance therefore falls from about 2.8x to
2.3x. Part of what looked like skill across the full test set was recognising
the 1,007 orders from markets that have never recorded a return at all.

That is the honest read on the geography feature. It is comfortably the
strongest input the model has, and a large share of its strength is a
recording gap rather than anything a customer did.

## 12. Turning the score into a decision

A classifier doesn't ship with a threshold. 0.5 is a default, not a decision.
The right cut-off is the one whose trade-off matches what a missed return and
a wasted review actually cost.

In [27]:
cp.threshold_sweep_plot(
    y_test, lgbm_score, optimise="f1",
    title="Precision / Recall Across Decision Cut-offs",
)

Precision decays from the left almost as fast as recall builds, which is the
characteristic shape of a weak ranker on a rare event. There is no cut-off
where both are comfortable at once.

The F1-optimal point marked on the chart is a mathematical convenience rather
than a business answer: it weights precision and recall equally, and the store
does not. What matters is the cost of one wasted review against the cost of
one missed return, and that ratio is a policy decision nobody has made yet.
The table below reframes the same trade-off in the units an operations manager
plans in: how many orders land in the queue, and how many reviews it takes to
catch one return.

In [28]:
def operating_point(flag_rate: float) -> dict:
    """Precision, recall and reviews-per-catch for flagging the top `flag_rate` share by score."""
    cut = np.quantile(lgbm_score, 1 - flag_rate)
    flagged = lgbm_score >= cut
    caught = int((flagged & (y_test == 1)).sum())
    n_flagged = int(flagged.sum())
    return {
        "flag_rate": flag_rate,
        "cut_off": cut,
        "orders_flagged": n_flagged,
        "returns_caught": caught,
        "recall": caught / int(y_test.sum()),
        "precision": caught / n_flagged if n_flagged else np.nan,
        "reviews_per_catch": n_flagged / caught if caught else np.nan,
    }

operating_table = pd.DataFrame(
    [operating_point(r) for r in [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]]
)
operating_table.round(3)

,flag_rate,cut_off,orders_flagged,returns_caught,recall,precision,reviews_per_catch
0,0.05,0.716,251,55,0.235,0.219,4.564
1,0.10,0.667,501,81,0.346,0.162,6.185
2,0.15,0.629,751,98,0.419,0.130,7.663
3,0.20,0.607,1002,112,0.479,0.112,8.946
4,0.25,0.585,1252,123,0.526,0.098,10.179
5,0.30,0.558,1502,140,0.598,0.093,10.729


Every extra point of recall costs precision, and the exchange rate gets worse
as the queue grows. Flagging the riskiest 5% of orders catches 23.5% of the
returns at 4.6 reviews per catch. Tripling the queue to 15% raises that to
41.9%, but each catch now costs 7.7 reviews. Doubling again to 30% reaches
59.8% recall at 10.7 reviews per catch, more than double the cost per catch of
the first tranche for well under three times the returns.

The first tranche is the only one where this model clearly earns its keep.
Reviewing 251 orders out of 5,007 to intercept 55 returns is a pilot a team
could staff next week. A 1,502-order queue to intercept 140 is a department,
and by then precision has fallen to 9%, barely above the 5.9% base rate of the
markets where returns actually happen.

In [29]:
# 5%: the only tranche where precision (21.9%) sits far enough above the
# base rate to be worth a reviewer's time, and small enough to staff.
CHOSEN_FLAG_RATE = 0.05
chosen = operating_point(CHOSEN_FLAG_RATE)
chosen_pred = (lgbm_score >= chosen["cut_off"]).astype(int)

cm = pd.DataFrame({"actual": y_test.map({0: "kept", 1: "returned"}),
                   "predicted": pd.Series(chosen_pred, index=y_test.index).map(
                       {0: "kept", 1: "returned"})})
cp.cross_tab_heatmap(
    cm, "actual", "predicted", normalize="row", show_counts=True,
    row_order=["kept", "returned"], col_order=["kept", "returned"],
    colorscale=cp.SEQ_BLUE,
    title=f"Confusion Matrix at the {CHOSEN_FLAG_RATE:.0%} Flag Rate (row = actual)",
)

Read the top row first. Of the 234 orders that did come back, the queue
catches 55 and misses 179. Of the 4,773 that did not, 196 are pulled in for a
review that finds nothing. Both error types are large, and the misses
outnumber the false alarms.

A queue like this is worth running only where a review is cheap and a caught
return is expensive. It is not a control that prevents returns. It is a
sampling rule that concentrates attention on orders where a return is about
four and a half times more likely than average, and it leaves three quarters
of returns to arrive unannounced.

## 13. What drives the prediction

SHAP on the full test set would take a while for little benefit past a few
thousand rows, so the beeswarm below samples 3,000 test orders.

In [30]:
shap_sample = X_test_c.sample(n=3000, random_state=42)
explainer = shap.TreeExplainer(lgbm)
shap_values = explainer.shap_values(shap_sample)
np.shape(shap_values)

D:\miniconda3\envs\analyst_313\Lib\site-packages\shap\explainers\_tree.py:632: UserWarning:

LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray



(3000, 16)

In [31]:
cp.shap_summary_plot(
    shap_values, shap_sample, top_n=15,
    title="SHAP Summary — Return-Risk Drivers",
)

One feature carries the model. Mean absolute SHAP for `market_region` is 0.75,
five times `quantity` at 0.14, with everything else below 0.11. The shape of
the plot makes the point without any numbers: a single wide band of geography
at the top, and beneath it a row of near-vertical stripes where the other
fifteen features barely move any prediction at all.

Among those minor features the directions are at least sensible. Larger orders
by quantity and by product count push risk up slightly, which fits the idea
that a bigger and more varied basket has more chances to contain something the
customer sends back. But these are small effects sitting underneath a
geography term that section 11 already showed to be part recording artefact,
and none is strong enough to build a rule on.

In [32]:
perm = permutation_importance(
    lgbm, X_test_c, y_test, n_repeats=10, scoring="average_precision", random_state=42,
)
importance_df = pd.DataFrame({
    "feature": np.repeat(X_test_c.columns, perm.importances.shape[1]),
    "drop": perm.importances.ravel(),
})
cp.grouped_bar_plot(
    importance_df, "feature", "drop", ci_method="bootstrap", min_n_flag=0,
    top_n=12, orientation="horizontal",
    title="Permutation Importance — Drop in Average Precision When a Feature is Shuffled",
)

Permutation importance agrees and is blunter about it. Shuffling
`market_region` costs 0.075 average precision, more than half the model's
total of 0.132. Shuffling `quantity` costs 0.021. Every other feature costs
less than 0.002, and `ship_lag_days` comes out marginally negative, meaning
the model does no worse without it.

The features the original notebook leaned on hardest, ship mode and priority
and segment, do nothing here. That matches the association tests run earlier,
which found no significant relationship between any of the three and whether
an order came back. Two independent methods pointing at the same empty set is
reasonably strong evidence that the order record does not describe why people
return things.

## 14. Bottom line

In [33]:
n_test = len(y_test)
n_returns_test = int(y_test.sum())
lgbm_ap = model_scores.loc["lightgbm", "average_precision"]
dummy_ap = model_scores.loc["dummy (most frequent)", "average_precision"]
train_test_gap = lgbm_split_scores.loc["train", "average_precision"] - lgbm_split_scores.loc["test", "average_precision"]

display(Markdown(f'''
**Returns in this data are close to unpredictable from what an order record
holds.** The best model reaches {lgbm_ap:.3f} average precision against a
{dummy_ap:.3f} base rate, on {n_test:,} held-out orders containing
{n_returns_test} returns. That is about three times better than chance at
ranking, with a train-to-test gap of {train_test_gap:.3f}, so it is a real
ceiling rather than an overfit. It is also nowhere near good enough to trust
on any individual order.

Three findings sit behind that number and matter more than it does.

**The strongest signal is a recording gap, not behaviour.** Three of the seven
markets, Africa and EMEA and Canada, covering 4,893 orders, have no recorded
returns whatsoever. Not a low rate: zero, across four years. `market_region`
is the model's dominant feature by a factor of five over the next one, and
much of what it contributes is recognising those markets. Restricted to the
four markets that do record returns, lift over chance drops from 2.8x to 2.3x.

**A linear model matches the gradient-boosted tree.** Weighted logistic
regression scores 0.139 average precision against the booster's
{lgbm_ap:.3f}, and the bootstrap interval on that difference straddles zero.
Ninety-eight rounds of boosting with native categorical splits find nothing a
set of one-hot coefficients missed. That is what it looks like when there is
no structure left to extract.

**The order record does not describe why people return things.** Ship mode,
priority and segment show no significant association with returning, and
permutation importance puts all three at effectively zero.

**What to do with this.** Do not put an automated flag into production on it.
The 5% queue is defensible as a pilot precisely because it is small and
reversible, and because its real value is the labelled review outcomes it
generates rather than the catches themselves. If returns are worth attacking
properly, the missing inputs are the ones a returns process produces and this
warehouse does not carry: per-product return history, per-customer return
history, and reason codes.

Before any of that, the recording gap needs an owner. A market reporting zero
returns across four years is a data-collection question, not a fact about
customers, and every model built on this table inherits it.
'''))


**Returns in this data are close to unpredictable from what an order record
holds.** The best model reaches 0.132 average precision against a
0.047 base rate, on 5,007 held-out orders containing
234 returns. That is about three times better than chance at
ranking, with a train-to-test gap of 0.032, so it is a real
ceiling rather than an overfit. It is also nowhere near good enough to trust
on any individual order.

Three findings sit behind that number and matter more than it does.

**The strongest signal is a recording gap, not behaviour.** Three of the seven
markets, Africa and EMEA and Canada, covering 4,893 orders, have no recorded
returns whatsoever. Not a low rate: zero, across four years. `market_region`
is the model's dominant feature by a factor of five over the next one, and
much of what it contributes is recognising those markets. Restricted to the
four markets that do record returns, lift over chance drops from 2.8x to 2.3x.

**A linear model matches the gradient-boosted tree.** Weighted logistic
regression scores 0.139 average precision against the booster's
0.132, and the bootstrap interval on that difference straddles zero.
Ninety-eight rounds of boosting with native categorical splits find nothing a
set of one-hot coefficients missed. That is what it looks like when there is
no structure left to extract.

**The order record does not describe why people return things.** Ship mode,
priority and segment show no significant association with returning, and
permutation importance puts all three at effectively zero.

**What to do with this.** Do not put an automated flag into production on it.
The 5% queue is defensible as a pilot precisely because it is small and
reversible, and because its real value is the labelled review outcomes it
generates rather than the catches themselves. If returns are worth attacking
properly, the missing inputs are the ones a returns process produces and this
warehouse does not carry: per-product return history, per-customer return
history, and reason codes.

Before any of that, the recording gap needs an owner. A market reporting zero
returns across four years is a data-collection question, not a fact about
customers, and every model built on this table inherits it.
